# 2층 신경망 역전파: 수식 → 손계산 → NumPy 검증

입력 2 → 은닉 2 → 출력 1. 열벡터 관례로 계산한다.
원문에서 누락된 정답은 참고 예제와 동일하게 $y=1$로 명시한다.
모든 log는 자연로그, 배치 없이 단일 표본의 BCE를 사용한다.
자동미분은 사용하지 않는다.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
(ROOT / "outputs").mkdir(exist_ok=True)
np.random.seed(42)
np.set_printoptions(precision=10, suppress=True)

## 1. 고정 입력과 shape

| 기호 | shape | 값 |
|---|---|---|
| $x$ | (2,) | [1,0] |
| $W_1$ | (2,2) | [[0.1,0.2],[0.3,0.4]] |
| $b_1$ | (2,) | [0,0] |
| $W_2$ | (2,) | [0.5,0.6] |
| $b_2,y$ | scalar () | 0, 1 |

$W_1$의 행은 은닉 뉴런, 열은 입력 성분이다.
NumPy 1차원 배열에서는 전치보다 연산 관례와 outer를 명확히 적는다.

## 2. 순전파 수식과 손계산

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad
\sigma'(z)=\sigma(z)(1-\sigma(z)).$$

$$z_1=W_1x+b_1=
[0.1\cdot1+0.2\cdot0,\;0.3\cdot1+0.4\cdot0]
=[0.1,0.3]\quad(2,).$$

$$a_1=\sigma(z_1)=
[1/(1+e^{-0.1}),1/(1+e^{-0.3})]
\approx[0.5249791875,0.5744425168]\quad(2,).$$

$$z_2=W_2a_1+b_2
=0.5(0.5249791875)+0.6(0.5744425168)
\approx0.6071551038\quad().$$

$$\hat y=\sigma(z_2)\approx0.6472915700\quad().$$
$$L=-[y\log\hat y+(1-y)\log(1-\hat y)]
=-\log(0.6472915700)\approx0.4349584369\quad().$$

표시는 반올림하지만 다음 단계 계산에는 충분한 정밀도의 값을 유지한다.

## 3. 연쇄 법칙을 출력에서 입력 방향으로 전개

**① 손실 → 예측 확률 (scalar)**
$$\frac{\partial L}{\partial\hat y}
=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}
=-\frac1{0.6472915700}\approx-1.5448988468\quad().$$

**② 예측 확률 → 출력 logit (scalar, 필수 체크포인트)**
$$\delta_2=\frac{\partial L}{\partial z_2}
=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)
=\hat y-y\approx-0.3527084300\quad().$$

**③ 출력 가중치와 편향**
$$\frac{\partial L}{\partial W_2}
=\delta_2a_1
\approx[-0.1851645850,-0.2026107182]\quad(2,),$$
$$\frac{\partial L}{\partial b_2}
=\delta_2\approx-0.3527084300\quad().$$

**④ 은닉 활성화로 전파**
$$\frac{\partial L}{\partial a_1}
=\delta_2W_2
\approx[-0.1763542150,-0.2116250580]\quad(2,).$$

**⑤ 은닉 logit으로 전파 (필수 체크포인트)**
$$\delta_1=\frac{\partial L}{\partial z_1}
=\frac{\partial L}{\partial a_1}\odot a_1\odot(1-a_1).$$
$$\delta_{1,1}=(-0.1763542150)(0.5249791875)(0.4750208125)
\approx-0.0439785158,$$
$$\delta_{1,2}=(-0.2116250580)(0.5744425168)(0.4255574832)
\approx-0.0517335044.$$
$$\delta_1\approx[-0.0439785158,-0.0517335044]\quad(2,).$$

**⑥ 입력 가중치와 편향**
$$\frac{\partial L}{\partial W_1}
=\delta_1x^T
=\operatorname{outer}(\delta_1,x)
\approx\begin{bmatrix}-0.0439785158&0\\-0.0517335044&0\end{bmatrix}
\quad(2,2),$$
$$\frac{\partial L}{\partial b_1}=\delta_1\quad(2,).$$

추가로 $\partial L/\partial x=W_1^T\delta_1$이며 shape은 (2,)이다.
두 번째 입력이 0이므로 $W_1$ 두 번째 열의 기울기가 0인 점도 확인한다.
원문 참고값 -0.0513은 계산 오류이며 올바른 4자리 값은 -0.0517이다.

In [2]:
from src.backprop import fixed_example, forward, backward
p = fixed_example()
fwd = forward(**p)
grad = backward(**p)
for name, value in fwd.items():
    print(f"{name:12s} shape={np.shape(value)!s:8s} value={value}")
for name, value in grad.items():
    print(f"dL/{name:9s} shape={np.shape(value)!s:8s} value={value}")
print("dL/x", p["W1"].T @ grad["d_z1"])

z1           shape=(2,)     value=[0.1 0.3]
a1           shape=(2,)     value=[0.5249791875 0.5744425168]
z2           shape=()       value=0.6071551038264654
y_pred       shape=()       value=0.6472915699946178
loss         shape=()       value=0.43495843685146063
dL/d_y_pred  shape=()       value=-1.5448988467566709
dL/d_z2      shape=()       value=-0.3527084300053822
dL/W2        shape=(2,)     value=[-0.185164585  -0.2026107182]
dL/b2        shape=()       value=-0.3527084300053822
dL/d_a1      shape=(2,)     value=[-0.176354215 -0.211625058]
dL/d_z1      shape=(2,)     value=[-0.0439785158 -0.0517335044]
dL/W1        shape=(2, 2)   value=[[-0.0439785158 -0.          ]
 [-0.0517335044 -0.          ]]
dL/b1        shape=(2,)     value=[-0.0439785158 -0.0517335044]
dL/x [-0.0199179029 -0.0294891049]


## 4. 독립적으로 적은 손계산 상수와 소수점 4자리 비교

아래 기대값은 수식에 숫자를 대입해 얻은 상수다.
코드 출력에서 기대값을 재생성하지 않는다.
모든 순전파 중간값과 필수 역전파 항목, 편향을 비교한다.

In [3]:
hand_forward = {
    "z1": [0.1, 0.3],
    "a1": [0.5249791875, 0.5744425168],
    "z2": 0.6071551038,
    "y_pred": 0.6472915700,
    "loss": 0.4349584369,
}
hand_backward = {
    "d_y_pred": -1.5448988468,
    "d_z2": -0.3527084300,
    "W2": [-0.1851645850, -0.2026107182],
    "b2": -0.3527084300,
    "d_a1": [-0.1763542150, -0.2116250580],
    "d_z1": [-0.0439785158, -0.0517335044],
    "W1": [[-0.0439785158, 0], [-0.0517335044, 0]],
    "b1": [-0.0439785158, -0.0517335044],
}
from IPython.display import Markdown, display
comparison_rows = ["| 항목 | shape | 손계산 (4자리) | NumPy (4자리) | 일치 |",
                   "|---|---|---|---|---|"]
for computed, expected in [(fwd, hand_forward), (grad, hand_backward)]:
    for name, hand_value in expected.items():
        np.testing.assert_array_equal(
            np.round(computed[name], 4), np.round(hand_value, 4))
        print(f"PASS {name}: {np.round(computed[name], 4)}")
        manual = np.array2string(np.asarray(hand_value), precision=4, floatmode="fixed").replace(chr(10), " ")
        actual = np.array2string(np.asarray(computed[name]), precision=4, floatmode="fixed").replace(chr(10), " ").replace("-0.0000", "0.0000")
        comparison_rows.append(
            f"| {name} | {np.shape(computed[name])} | {manual} | {actual} | PASS |")
display(Markdown(chr(10).join(comparison_rows)))
print("All forward/backward hand calculations agree to 4 decimal places.")

PASS z1: [0.1 0.3]
PASS a1: [0.525  0.5744]
PASS z2: 0.6072
PASS y_pred: 0.6473
PASS loss: 0.435
PASS d_y_pred: -1.5449
PASS d_z2: -0.3527
PASS W2: [-0.1852 -0.2026]
PASS b2: -0.3527
PASS d_a1: [-0.1764 -0.2116]
PASS d_z1: [-0.044  -0.0517]
PASS W1: [[-0.044  -0.    ]
 [-0.0517 -0.    ]]
PASS b1: [-0.044  -0.0517]


| 항목 | shape | 손계산 (4자리) | NumPy (4자리) | 일치 |
|---|---|---|---|---|
| z1 | (2,) | [0.1000 0.3000] | [0.1000 0.3000] | PASS |
| a1 | (2,) | [0.5250 0.5744] | [0.5250 0.5744] | PASS |
| z2 | () | 0.6072 | 0.6072 | PASS |
| y_pred | () | 0.6473 | 0.6473 | PASS |
| loss | () | 0.4350 | 0.4350 | PASS |
| d_y_pred | () | -1.5449 | -1.5449 | PASS |
| d_z2 | () | -0.3527 | -0.3527 | PASS |
| W2 | (2,) | [-0.1852 -0.2026] | [-0.1852 -0.2026] | PASS |
| b2 | () | -0.3527 | -0.3527 | PASS |
| d_a1 | (2,) | [-0.1764 -0.2116] | [-0.1764 -0.2116] | PASS |
| d_z1 | (2,) | [-0.0440 -0.0517] | [-0.0440 -0.0517] | PASS |
| W1 | (2, 2) | [[-0.0440  0.0000]  [-0.0517  0.0000]] | [[-0.0440 0.0000]  [-0.0517 0.0000]] | PASS |
| b1 | (2,) | [-0.0440 -0.0517] | [-0.0440 -0.0517] | PASS |

All forward/backward hand calculations agree to 4 decimal places.


## 5. 중심차분으로 모든 학습 파라미터를 별도로 검증

$$\frac{\partial L}{\partial\theta_i}
\approx\frac{L(\theta+h e_i)-L(\theta-h e_i)}{2h},
\qquad h=10^{-5}.$$

이는 독립적인 검증이며 역전파 구현을 대신하지 않는다.
float64에서 모든 파라미터의 최대 절대오차가 $10^{-7}$ 이하여야 한다.

In [4]:
from src.calculus import numerical_gradient
for name in ["W1", "b1", "W2", "b2"]:
    def loss(candidate):
        """Compute loss with one parameter tensor replaced."""
        changed = dict(p)
        changed[name] = candidate
        return forward(**changed)["loss"]
    numerical = numerical_gradient(loss, p[name])
    error = np.max(np.abs(numerical - grad[name]))
    print(name, "shape", numerical.shape, "max absolute error", error)
    assert error < 1e-7

W1 shape (2, 2) max absolute error 3.695856121144203e-12
b1 shape (2,) max absolute error 3.695856121144203e-12
W2 shape (2,) max absolute error 3.2595592891482283e-12
b2 shape () max absolute error 1.6270873537393982e-12


## 해석

이 표본에서 출력 확률은 정답 1보다 작다. 출력 logit의 기울기가 음수이므로
경사하강법은 출력 logit을 키우는 방향으로 파라미터를 움직인다.
첫 번째 입력만 활성화되어 W1의 첫 번째 열만 직접 업데이트된다.
각 층은 행렬 연산을 하고 역전파는 이 연산들의 국소 미분을 연쇄 법칙으로 연결한다.